In [ ]:
import os
import sys
import json
import pandas as pd 
from os.path import abspath, join

from dotenv import load_dotenv

from langchain_community.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

sys.path.insert(0, abspath(join('..', '..')))
from mucap.rag_skeleton_patch import build_skeleton_text, annotation_consensus_summary


In [ ]:
DATASET_DIR = '../../dataset'
DATASET_DIR = abspath(DATASET_DIR)
DATASET_DIR

In [ ]:
class Document:
    def __init__(self, page_content, metadata):
        self.page_content = page_content
        self.metadata = metadata

In [ ]:
load_dotenv()
persist_directory = './ragdb_skeleton'

In [ ]:
JSON_FILENAME = '../../human_annotation/human_annotation_total.json'
with open(JSON_FILENAME, 'r') as file:
    data = json.load(file)
len(data)

In [ ]:
MUSIC_FEATURE_PATH = abspath(join('..', '..', 'music_features.csv'))
MUSIC_FEATURE_PATH

music_feature_df = pd.read_csv(MUSIC_FEATURE_PATH, encoding='utf-8')
music_feature_df

In [ ]:
docs = list()

for item in data:
    
    audio_url = item['data']['audio']
    audio_url_arr = audio_url.split('/')
    try:
        dataset, genre, filename = audio_url_arr[-3], audio_url_arr[-2], audio_url_arr[-1].replace('fixed_', '')
    except:
        print(f"audio_url_arr: {audio_url_arr}")
        continue
    json_path = join(DATASET_DIR, dataset, genre, f'{filename}.json')

    # if not exists:
    if os.path.exists(json_path) is False:
        print(f"Error audio_url: {audio_url}")
        continue
    
    
    # search row with filename (file_name in music_feature_df)
    feature = music_feature_df[music_feature_df['file_name'] == filename].iloc[0]

    
    top5_instruments = eval(feature['top5_instruments'])
    # collect the first item from top5_instruments
    top5_instruments = [item[0] for item in top5_instruments]
    top5_instruments = ', '.join(top5_instruments)
    
    music_feature_txt = (f"Music Feature: volume: {feature['volume']},\npitch: {feature['pitch']},\ntempo: {feature['tempo']},\n beat_frames: {feature['beat_frames']},\n"
               f"Top-5 instruments: {top5_instruments}.\n")
    
    
    gpt_json = json.load(open(json_path, 'r', encoding='utf-8'))


    music_feature_txt = gpt_json['input'][0]['text']
    try:
        music_feature_txt = music_feature_txt.split('[Context]')[1].split('volume(dB)')[0].strip()
    except IndexError:
        # remove the json
        os.remove(json_path)
        
        print(f"[Error] The [context] part of {music_feature_txt} is incorrect.")
        continue
    music_feeling_txt = gpt_json['output']['annotation']
 
    result = item['annotations'][0]['result']

    text_arr = list()

    for row in result:
        value = row['value']
        if 'text' not in value:
            continue
        
        text_arr.append(value['text'][0])

    annotation = ' '.join(text_arr)
    skeleton_txt = build_skeleton_text(feature)
    consensus_txt = annotation_consensus_summary(text_arr)
    row_txt = (
        f"Music Feature: {music_feature_txt}\n"
        f"{skeleton_txt}\n"
        f"{consensus_txt}\n"
        f"Music Feeling: {music_feeling_txt}\n"
        f"Annotation: {annotation}"
    )
    
    docs.append(Document(page_content=row_txt, 
                            metadata={
                                'source_path': audio_url,
                                'source_type': 'audio_caption',
                                'source': filename,
                                'dataset': dataset,
                                'genre': genre
                            })) 

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
vectordb = Chroma.from_documents(docs, embeddings, persist_directory=persist_directory)